# ReaderAdapt-Liver on LiTS17: ViT-UNETR segmenter + MedGemma 1.5 second reader (Colab, GPU)

**Runtime -> Change runtime type -> GPU** (T4 works; L4 / A100 is about 3x faster).

Two Colab secrets (key icon on the left, *Notebook access* on):

| Secret | Why | Where to get it |
|---|---|---|
| `KAGGLE_API_TOKEN` (or `KAGGLE_USERNAME` + `KAGGLE_KEY`) | download the two LiTS17 datasets | kaggle.com -> Settings -> API -> Create New Token |
| `HF_TOKEN` | download MedGemma 1.5 4B | huggingface.co -> Settings -> Access Tokens (read). First open https://huggingface.co/google/medgemma-1.5-4b-it and accept the Health AI Developer Foundations terms |

Pipeline: download LiTS (both Kaggle parts) -> notebook-compatible split (same as the ViT-UNETR liver notebook)
-> train the 3-class ViT-UNETR 2.5D -> segment + measure the 51 held-out CTs -> MedGemma second read ->
calibration (segmenter / MedGemma / fused) -> agentic first read + reports -> reader experiment -> figures ->
**website**.

Rough times on a T4: download 10-15 min, training 12 epochs ~60-90 min, segmentation ~10 min,
MedGemma ~1 min per case (~50 min). Set `QUICK = True` below for a ~40-case first look.

In [ ]:
# 1. Code + dependencies
REPO_URL = "https://github.com/AIVIETNAM-AIO-khaisiubeo/ReaderAdapt-Liver.git"
BRANCH = "lits-medgemma"          # use "main" once the pull request is merged
import os
if not os.path.exists("ReaderAdapt-Liver"):
    !git clone -q -b $BRANCH $REPO_URL
%cd ReaderAdapt-Liver
!git pull -q
!pip install -q -r requirements.txt -r requirements-deep.txt -r requirements-vlm.txt

In [ ]:
# 2. Secrets -> environment (KAGGLE_API_TOKEN / KAGGLE_USERNAME + KAGGLE_KEY, HF_TOKEN)
import os
from google.colab import userdata
for k in ("KAGGLE_API_TOKEN", "KAGGLE_USERNAME", "KAGGLE_KEY", "HF_TOKEN"):
    try:
        v = userdata.get(k)
        if v:
            os.environ[k] = v
    except Exception:
        pass
print({k: (k in os.environ) for k in ("KAGGLE_API_TOKEN", "KAGGLE_USERNAME", "KAGGLE_KEY", "HF_TOKEN")})
import torch
print(torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - change the runtime!")

In [ ]:
# 3. Run configuration
import yaml, torch
QUICK = False                     # True: proportional ~40-case subset, fewer epochs, for a fast first look
cfg = yaml.safe_load(open("configs/lits.yaml"))
ampere = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
cfg["segmenter"]["epochs"] = 30 if ampere else 12
if QUICK:
    cfg["run_name"], cfg["data"]["max_cases"], cfg["segmenter"]["epochs"] = "lits_quick", 40, 6
warm = "checkpoints/vitunetr_liver_2class.pt"   # optional, see checkpoints/README.md
cfg["segmenter"]["init_from"] = warm if os.path.exists(warm) else None
yaml.safe_dump(cfg, open("configs/lits_colab.yaml", "w"), sort_keys=False)
RUN = f"outputs/{cfg['run_name']}"
print("run:", RUN, "| epochs:", cfg["segmenter"]["epochs"], "| warm start:", cfg["segmenter"]["init_from"],
      "| MedGemma:", cfg["vlm"]["model_id"])

In [ ]:
# 4. LiTS17 from Kaggle (part 1 = segmentations + volume_pt1-5, part 2 = the remaining volumes), then the split
!python -m radagent download --config configs/lits_colab.yaml
!python -m radagent data --config configs/lits_colab.yaml

In [ ]:
# 5. Train the segmentation tool (3-class ViT-UNETR 2.5D: background / liver / tumour)
!python -m radagent train --config configs/lits_colab.yaml

In [ ]:
# 6. Segment + measure the held-out CTs (flip-TTA x MC-dropout passes, original resolution)
!python -m radagent infer --config configs/lits_colab.yaml

In [ ]:
# 7. MedGemma 1.5 second read of every held-out CT (downloads ~8 GB of weights the first time)
!python -m radagent vlm --config configs/lits_colab.yaml

In [ ]:
# 8. Calibration, agentic first read + reports, reader experiment, figures
!python -m radagent calibrate --config configs/lits_colab.yaml
!python -m radagent first-read --config configs/lits_colab.yaml
!python -m radagent simulate --config configs/lits_colab.yaml
!python -m radagent figures --config configs/lits_colab.yaml

In [ ]:
# 9. One agent trace and its report
!python -m radagent agent --config configs/lits_colab.yaml
import glob
print(open(sorted(glob.glob(f"{RUN}/reports/*.txt"))[0]).read())

In [ ]:
from IPython.display import Image, display
for f in ("ai_tools.png", "calibration_reliability.png", "segmentation_quality.png", "policy_comparison.png"):
    if os.path.exists(f"{RUN}/figures/{f}"):
        display(Image(f"{RUN}/figures/{f}", width=900))

## 10. Open the website

**Option A - straight from Colab (everything works, including *Recompute from the CT file*, because the CTs, the
GPU and the models are here).** The cell starts the website and prints a public `trycloudflare.com` link (a free
Cloudflare quick tunnel; anyone with the link can open it while the cell runs - stop the cell to close it).

In [ ]:
import subprocess, re, time
if not os.path.exists("cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared
app = subprocess.Popen(["python", "-m", "streamlit", "run", "app/streamlit_app.py", "--server.port", "8501",
                        "--server.headless", "true", "--", "--run", RUN],
                       stdout=open("streamlit.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8501", "--no-autoupdate"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in tunnel.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        print("Website:", m.group(0))
        break
try:
    while app.poll() is None:
        time.sleep(30)
except KeyboardInterrupt:
    pass
finally:
    tunnel.terminate(); app.terminate()

**Option B - on your own computer.** Download the run (without the training-slice cache), unzip it into the
`outputs/` folder of your local copy of the repository and start
`python -m streamlit run app/streamlit_app.py -- --run outputs/lits`. Everything except *Recompute from the CT file*
works without a GPU (recompute needs the LiTS files, PyTorch and, for MedGemma, a GPU).

In [ ]:
!cd outputs && zip -qr ../lits_run.zip {cfg['run_name']} -x "{cfg['run_name']}/cache/train_slices/*"
!ls -lh lits_run.zip
from google.colab import files
files.download("lits_run.zip")
# Optional: keep a copy in Google Drive
# from google.colab import drive; drive.mount("/content/drive"); !cp lits_run.zip /content/drive/MyDrive/